# **1) EDA & Cleaning Script**

**Project by Sneha, Olivia, and Riley**

This script include the initial EDA of scraped data, the cleaning process, our refinements and remapping of genres, new genre codes and dictionaries created, and new visuals created from the cleaned and recategorized data.

In [ ]:
#importing packages
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns


In [ ]:
# Mounting to Google Drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
path = '/content/drive/My Drive/DS 4002/Project 1/book_summaries_10k(in).csv'
book_df = pd.read_csv(path)

#Cleaning and reducing data frame to the correct columns
book_df = book_df[['Book Title', 'Genre', 'Synopsis']]
book_df = book_df.dropna()

#Separating the listed genres out
book_df['genre_list'] = book_df['Genre'].apply(lambda x: [g.strip() for g in str(x).split(',') if g.strip()])
book_df['num_genres'] = book_df['genre_list'].apply(len)
display(book_df.head(10))

FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/My Drive/DS 4002/Project 1/book_summaries_10k(in).csv'

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
#Listing all the genres (to see overarching or all-encapsulating genres to cut) and make a bar chart of the top ten genres by frequency of titles.
all_genres = [g for sublist in book_df['genre_list'] for g in sublist]
genre_counts = pd.Series(all_genres).value_counts()
unique_genres = genre_counts.index.tolist()
print('Number of unique genres:', len(unique_genres))
print('Unique genres:', unique_genres)

NameError: name 'book_df' is not defined

In [ ]:
#display all genre_counts
pd.set_option('display.max_rows', None)
display(genre_counts)

In [ ]:
#Limitation: 'Fiction', 'Speculative Fiction', 'Novel' are getting being counted as a genre. All the top ten genres are fiction genres.
#Many titles will list fiction along with the genre

#Make a bar chart of the genre occurences
top_10_genres = genre_counts.head(10).index
plt.figure()
sns.barplot(x=top_10_genres, y=genre_counts.loc[top_10_genres])
plt.xticks(rotation=45)
plt.xlabel('Genre')
plt.ylabel('Number of Titles in Each Genre')
plt.title('Top 10 Genres by Number of Titles')
plt.show()

In [ ]:
unique_genres = sorted(list(set(g for sublist in book_df['genre_list'] for g in sublist if g != 'nan')))

# Co-occurrence matrix
co_occurrence_matrix = pd.DataFrame(0, index=unique_genres, columns=unique_genres)

for index, row in book_df.iterrows():
    genres_in_book = [g for g in row['genre_list'] if g != 'nan']
    for g1 in genres_in_book:
        for g2 in genres_in_book:
            co_occurrence_matrix.loc[g1, g2] += 1

filtered_matrix = co_occurrence_matrix.loc[top_10_genres, top_10_genres]

plt.figure(figsize=(12, 10))
sns.heatmap(filtered_matrix, annot=True, fmt="d", cmap="Reds", linewidths=.5)
plt.title('Heatmap by Number of Genre Co-Occurrences')
plt.xlabel('Genre 2')
plt.ylabel('Genre 1')
plt.show()

In [ ]:
#Issue of Co-occurance with "Fiction" and "Speculative Fiction" and "Novel" with other categories above. Certain genres have too few titles as well.
#Difficult data cleaning choices--> Reducing complexity in genres in order to improve model accuracy.
#Certain genres had too few titles or were too niche. The code below is used to create keys for new genre allocations and this will be used to train the model.
#We found this to be primarily non-fiction titles getting dropped which means this sorter will lean heavily towards fiction.

#Anthropic. (2026). Claude Sonnet 5 [Large language model]. https://claude.ai/ Artificial intelligence was used to assist with code development of this sections and debugging.


Target_Genres = [
    'Science Fiction', 'Fantasy', "Children's literature", 'Young adult literature',
    'Mystery', 'Historical fiction', 'Suspense', 'Crime Fiction', 'Horror', 'Thriller',
    'Romance novel', 'Detective fiction', 'Adventure', 'Humor',
    'Utopian and dystopian fiction', 'Spy fiction', 'Alternate history',
    'Literary fiction', 'Satire', 'Autobiographical novel', 'Gothic fiction',
    'War novel', 'High fantasy', 'Apocalyptic and post-apocalyptic fiction',
    'Techno-thriller', 'Bildungsroman', 'Western', 'Steampunk',
]


Merged_Genres = [
    (['Science Fiction'], [
        'Military science fiction', 'Hard science fiction', 'Soft science fiction',
        'Social science fiction', 'Feminist science fiction', 'Comic science fiction',
        'Space opera', 'Planetary romance', 'Sword and planet', 'Space western',
        'Time travel', 'Cyberpunk', 'Postcyberpunk', 'Scientific romance',
        'Edisonade', 'Future history', 'Invasion literature']),
    (['Fantasy'], [
        'Sword and sorcery', 'Dark fantasy', 'Urban fantasy', 'Contemporary fantasy',
        'Historical fantasy', 'Comic fantasy', 'Science fantasy', 'Dying Earth subgenre',
        'Fantasy of manners', 'Bangsian fantasy', 'New Weird', 'Wuxia', 'Fairy tale',
        'Fable', 'Chivalric romance', 'Medieval romance']),
    (["Children's literature"], [
        'School story', "Boys' school stories", 'English public-school stories']),
    (["Children's literature", 'Fantasy'], ['Juvenile fantasy']),
    (['Young adult literature'], ['Youth']),
    (['Historical fiction'], ['Historical novel', 'Biographical novel']),
    (['Historical fiction', 'Detective fiction'], ['Historical whodunnit']),
    (['Romance novel', 'Historical fiction'], [
        'Historical romance', 'Regency romance', 'Georgian romance', 'Elizabethan romance']),
    (['Romance novel'], ['Paranormal romance', 'Bit Lit', 'Chick lit', 'Erotica']),
    (['Romance novel', 'Humor'], ['Romantic comedy']),
    (['Horror'], [
        'Supernatural', 'Fantastique', 'Ghost story', 'Vampire fiction',
        'Zombie', 'Zombies in popular culture']),
    (['Thriller'], ['Conspiracy fiction']),
    (['Detective fiction'], ['Hardboiled', 'Whodunit', 'Locked room mystery', 'Cozy']),
    (['Adventure'], [
        'Adventure novel', 'Lost World', 'Robinsonade', 'Sea story', 'Naval adventure']),
    (['Utopian and dystopian fiction'], ['Dystopia', 'Utopian fiction']),
    (['Literary fiction'], [
        'Picaresque novel', 'Psychological novel', 'Parallel novel', 'Social novel',
        'Industrial novel', 'Magic realism', 'Absurdist fiction', 'Campus novel']),
    (['Humor'], [
        'Comedy', 'Humour', 'Comic novel', 'Black comedy', 'Parody', 'Farce',
        'Tragicomedy', 'Comedy of manners']),
    (['Autobiographical novel'], ['Roman à clef']),
    (['Gothic fiction'], ['American Gothic Fiction']),
    (['Apocalyptic and post-apocalyptic fiction'], [
        'Post-holocaust', 'Catastrophic literature']),
    (['Bildungsroman'], ['Coming of age', 'Künstlerroman']),
    (['Western'], ['Western fiction']),
]

Genre_Map = {}
for targets, sources in Merged_Genres:
    for src in sources:
        Genre_Map.setdefault(src, [])
        for t in targets:
            if t not in Genre_Map[src]:
                Genre_Map[src].append(t)
for g in Target_Genres:
    Genre_Map.setdefault(g, [g])

#Handling for mistakes and dropping empty values
bad = {t for ts in Genre_Map.values() for t in ts} - set(Target_Genres)
assert not bad, f'Unknown target labels in rules: {bad}'

#Remapping Function
def remap_genres(genres):
    """Map one book's genre list to the target labels, deduplicated, order kept."""
    if not isinstance(genres, (list, tuple, np.ndarray)):
        return []
    out = []
    for g in genres:
        for t in Genre_Map.get(str(g).strip(), []):
            if t not in out:
                out.append(t)
    return out

book_df['genre_mapped'] = book_df['genre_list'].apply(remap_genres)

#Ensuring no empty columns.
orig = book_df['genre_list'].explode().dropna().astype(str).str.strip()
dropped_counts = orig[~orig.isin(Genre_Map.keys())].value_counts()
print('Dropped labels (top 25):')
print(dropped_counts.head(25))

no_label = book_df['genre_mapped'].str.len() == 0
print(f'\nBooks with no genre after mapping: {no_label.sum()} of {len(book_df)} '
      f'({no_label.mean():.1%})')

mapped_counts = book_df['genre_mapped'].explode().value_counts()
print('\nTitles per genre after mapping:')
print(mapped_counts)

#Creating new data frame that can be used for new scripts and charts further into the investigation.
cleaned_book_df = pd.DataFrame({
    'Book Title': book_df['Book Title'],
    'Genre': book_df['genre_mapped'],
    'Synopsis': book_df['Synopsis']})

#Additional cleaning for empty genre columns
initial_rows = len(cleaned_book_df)
cleaned_book_df = cleaned_book_df[cleaned_book_df['Genre'].apply(lambda x: len(x) > 0)]
dropped_rows = initial_rows - len(cleaned_book_df)

print(f"Dropped {dropped_rows} rows with empty genre lists.")
print("DataFrame after removing empty genres (first 10 rows):")
display(cleaned_book_df.head(10))


In [ ]:
#Displaying data dictionary of new reorganized genres
unique_new_genres = sorted(list(set(g for sublist in book_df['genre_mapped'] for g in sublist if g != 'nan')))
genre_counts_new = pd.Series([g for sublist in book_df['genre_mapped'] for g in sublist if g != 'nan']).value_counts()
top_10_new_genres = genre_counts_new.head(10).index
print('Number of unique genres:', len(unique_new_genres))
print('Unique genres:', unique_new_genres)

#Creating bar chart that displays unique genres that are occuring the most frequently, uding the new cleaned data frame.
plt.figure(figsize = (10, 6))
sns.barplot(x=top_10_new_genres, y=genre_counts_new.loc[top_10_new_genres])
plt.xticks(rotation=45, ha='right') # Added ha='right' for better alignment
plt.xlabel('Genre')
plt.ylabel('Number of Titles in Each Genre')
plt.title('Top 10 Genres by Number of Titles')
plt.show()

In [ ]:
#Displaying a New Heatmap of the cleaned DF
#Creating co-occurance matrix
unique_genres = sorted(list(set(g for sublist in cleaned_book_df['Genre'] for g in sublist if g != 'nan')))
new_co_occurrence_matrix = pd.DataFrame(0, index=unique_genres, columns=unique_genres)
for index, row in cleaned_book_df.iterrows():
    genres_in_book = [g for g in row['Genre'] if g != 'nan']
    for g1 in genres_in_book:
        for g2 in genres_in_book:
            new_co_occurrence_matrix.loc[g1, g2] += 1

filtered_matrix_cleaned = new_co_occurrence_matrix.loc[top_10_new_genres, top_10_new_genres]

#Creating the Heatmap
plt.figure(figsize=(12, 10))
sns.heatmap(filtered_matrix_cleaned, annot=True, fmt="d", cmap="Reds", linewidths=.5)
plt.title('Heatmap by Number of Genre Co-Occurrences (Cleaned Data)')
plt.xlabel('Genre 2')
plt.ylabel('Genre 1')
plt.show()

In [ ]:
# Downloading new data to use in new scripts and for the modeling.
cleaned_book_df.assign(Genre=cleaned_book_df['Genre'].str.join(', ')) \
               .to_csv('cleaned_book_data.csv', index=False)